In [7]:
from pathlib import Path
import math

import cairosvg
from PIL import Image


# ============================================================
# Output
# ============================================================

OUTDIR = Path("imasen_phi_logo_v7")
OUTDIR.mkdir(exist_ok=True)


# ============================================================
# Canvas
# ============================================================

SIZE = 1024

CX = 520
CY = 525


# ============================================================
# Ellipse geometry
# ============================================================

# Horizontal radius
ORBIT_RX = 425

# Flattening ratio:
#
# 1.00 = circle
# 0.70 = mildly flattened
# 0.58 = current
# 0.45 = strongly flattened
# 0.35 = very flat
#
ORBIT_FLATTEN = 0.48

# Vertical radius is derived automatically
ORBIT_RY = ORBIT_RX * ORBIT_FLATTEN

# Visual tilt
ROTATION = -8


# ============================================================
# Pure colors
# ============================================================

GOLD = "#F0BE32"
BLUE = "#24409A"
WHITE = "#FFFFFF"


# ============================================================
# Main proportions
# ============================================================

I_STEM = 112

DOT_DIAMETER = 105


# Orbit widths
ORBIT_W_MIN = 24
ORBIT_W_WAVE_END = 40
ORBIT_W_MAX = 72


# White underlay thickness
ORBIT_BORDER = 14

I_BORDER = 18
DOT_BORDER = 16


# ============================================================
# Angular convention
# ============================================================
#
# Positive angle = visually counterclockwise
#
# 0°   = right
# 90°  = top
# 180° = left
# 270° = bottom
#
# ============================================================


# Gold node
DOT_ANGLE = 189.0


# Wave starts 15° counterclockwise from node
WAVE_OFFSET_FROM_DOT = 45.0

WAVE_START_ANGLE = (
    DOT_ANGLE
    + WAVE_OFFSET_FROM_DOT
)


# Wave continues counterclockwise
WAVE_END_ANGLE = 300.0


# Maximum orbit width around upper-right
WIDTH_PEAK_ANGLE = 30.0


# ============================================================
# Fourfold damped waveform
# ============================================================

WAVE_CYCLES = 4.0

WAVE_AMPLITUDE = 72

WAVE_DECAY = 1.70


# Short smooth transition at wave start
WAVE_RISE_FRAC = 0.03

# Short smooth transition back to ellipse
WAVE_FADE_FRAC = 0.08


# ============================================================
# Wave direction
# ============================================================
#
# -1.0:
# first visible lobe goes UP,
# then DOWN
#
# +1.0:
# first visible lobe goes DOWN,
# then UP
#
WAVE_DIRECTION = -1.0


# ============================================================
# Layering
# ============================================================

# Orbit below this line is redrawn in front of I
FRONT_CLIP_Y = 560


# ============================================================
# Geometry utilities
# ============================================================

def rotate_point(
    x,
    y,
    cx,
    cy,
    angle_deg,
):

    a = math.radians(
        angle_deg
    )

    dx = x - cx
    dy = y - cy

    return (
        cx
        + dx * math.cos(a)
        - dy * math.sin(a),

        cy
        + dx * math.sin(a)
        + dy * math.cos(a),
    )


def rotate_vector(
    vx,
    vy,
    angle_deg,
):

    a = math.radians(
        angle_deg
    )

    return (
        vx * math.cos(a)
        - vy * math.sin(a),

        vx * math.sin(a)
        + vy * math.cos(a),
    )


def normalize(
    vx,
    vy,
):

    length = math.hypot(
        vx,
        vy,
    )

    if length == 0:

        return (
            0.0,
            0.0,
        )

    return (
        vx / length,
        vy / length,
    )


def smoothstep(t):

    t = max(
        0.0,
        min(
            1.0,
            t,
        ),
    )

    return (
        t
        * t
        * (
            3
            - 2 * t
        )
    )


def interpolate(
    a,
    b,
    t,
):

    return (
        a
        + (
            b - a
        )
        * smoothstep(t)
    )


# ============================================================
# Ellipse
# ============================================================

def ellipse_point(
    angle_deg,
):
    """
    Increasing angle corresponds to
    visually counterclockwise motion.
    """

    theta = math.radians(
        angle_deg
    )

    x = (
        CX
        + ORBIT_RX
        * math.cos(theta)
    )

    y = (
        CY
        - ORBIT_RY
        * math.sin(theta)
    )

    return rotate_point(
        x,
        y,
        CX,
        CY,
        ROTATION,
    )


def ellipse_tangent(
    angle_deg,
):

    theta = math.radians(
        angle_deg
    )

    dx = (
        -ORBIT_RX
        * math.sin(theta)
    )

    dy = (
        -ORBIT_RY
        * math.cos(theta)
    )

    return rotate_vector(
        dx,
        dy,
        ROTATION,
    )


def ellipse_outward_normal(
    angle_deg,
):

    x, y = ellipse_point(
        angle_deg
    )

    tx, ty = normalize(
        *ellipse_tangent(
            angle_deg
        )
    )


    # Candidate normal
    nx = -ty
    ny = tx


    # Radial direction from center
    rx = x - CX
    ry = y - CY


    # Ensure outward direction
    if (
        nx * rx
        + ny * ry
        < 0
    ):

        nx = -nx
        ny = -ny


    return (
        nx,
        ny,
    )


# ============================================================
# Variable orbit width
# ============================================================

def orbit_width(
    angle_deg,
):

    progress = (
        angle_deg
        - DOT_ANGLE
    ) % 360


    wave_start_progress = (
        WAVE_START_ANGLE
        - DOT_ANGLE
    ) % 360


    wave_end_progress = (
        WAVE_END_ANGLE
        - DOT_ANGLE
    ) % 360


    peak_progress = (
        WIDTH_PEAK_ANGLE
        - DOT_ANGLE
    ) % 360


    # --------------------------------
    # Node -> wave start
    # --------------------------------

    if (
        progress
        <= wave_start_progress
    ):

        return (
            ORBIT_W_MIN
        )


    # --------------------------------
    # Wave region
    # --------------------------------

    if (
        progress
        <= wave_end_progress
    ):

        t = (
            progress
            - wave_start_progress
        ) / (
            wave_end_progress
            - wave_start_progress
        )

        return interpolate(
            ORBIT_W_MIN,
            ORBIT_W_WAVE_END,
            t,
        )


    # --------------------------------
    # Wave end -> width maximum
    # --------------------------------

    if (
        progress
        <= peak_progress
    ):

        t = (
            progress
            - wave_end_progress
        ) / (
            peak_progress
            - wave_end_progress
        )

        return interpolate(
            ORBIT_W_WAVE_END,
            ORBIT_W_MAX,
            t,
        )


    # --------------------------------
    # Width maximum -> node
    # --------------------------------

    t = (
        progress
        - peak_progress
    ) / (
        360
        - peak_progress
    )

    return interpolate(
        ORBIT_W_MAX,
        ORBIT_W_MIN,
        t,
    )


# ============================================================
# Wave amplitude envelope
# ============================================================

def wave_envelope(t):
    """
    First full peak is the largest peak.

    Only a very short smooth ramp is used
    at the beginning to keep geometric continuity.
    """

    rise = smoothstep(
        t
        / WAVE_RISE_FRAC
    )


    decay = math.exp(
        -WAVE_DECAY * t
    )


    fade_out = smoothstep(
        (
            1 - t
        )
        / WAVE_FADE_FRAC
    )


    return (
        rise
        * decay
        * fade_out
    )


# ============================================================
# Ordinary ellipse sampling
# ============================================================

def sample_arc(
    angle_start,
    angle_end,
    n,
    include_first=True,
):

    points = []
    widths = []


    start_index = (
        0
        if include_first
        else 1
    )


    for i in range(
        start_index,
        n,
    ):

        t = (
            i
            / (n - 1)
        )


        angle = (
            angle_start
            + (
                angle_end
                - angle_start
            )
            * t
        )


        points.append(
            ellipse_point(
                angle
            )
        )


        widths.append(
            orbit_width(
                angle
            )
        )


    return (
        points,
        widths,
    )


# ============================================================
# Wave sampling
# ============================================================

def sample_wave(
    n=420,
    include_first=False,
):

    points = []
    widths = []


    start_index = (
        0
        if include_first
        else 1
    )


    for i in range(
        start_index,
        n,
    ):

        t = (
            i
            / (n - 1)
        )


        angle = (
            WAVE_START_ANGLE
            + (
                WAVE_END_ANGLE
                - WAVE_START_ANGLE
            )
            * t
        )


        # Base ellipse
        x, y = ellipse_point(
            angle
        )


        # Local outward normal
        nx, ny = (
            ellipse_outward_normal(
                angle
            )
        )


        # Four oscillations
        phase = (
            2
            * math.pi
            * WAVE_CYCLES
            * t
        )


        # Negative sign means:
        #
        # first lobe UP
        # second lobe DOWN
        #
        amp = (
            WAVE_DIRECTION
            * WAVE_AMPLITUDE
            * wave_envelope(t)
            * math.sin(phase)
        )


        x += (
            nx
            * amp
        )

        y += (
            ny
            * amp
        )


        points.append(
            (
                x,
                y,
            )
        )


        widths.append(
            orbit_width(
                angle
            )
        )


    return (
        points,
        widths,
    )


# ============================================================
# Build continuous centerline
# ============================================================

def build_centerline():

    # Node -> wave start
    pre_wave_points, pre_wave_widths = (
        sample_arc(
            DOT_ANGLE,
            WAVE_START_ANGLE,
            n=80,
            include_first=True,
        )
    )


    # Fourfold damped wave
    wave_points, wave_widths = (
        sample_wave(
            n=420,
            include_first=False,
        )
    )


    # Wave end -> node
    post_wave_points, post_wave_widths = (
        sample_arc(
            WAVE_END_ANGLE,
            DOT_ANGLE + 360,
            n=700,
            include_first=False,
        )
    )


    points = (
        pre_wave_points
        + wave_points
        + post_wave_points
    )


    widths = (
        pre_wave_widths
        + wave_widths
        + post_wave_widths
    )


    return (
        points,
        widths,
    )


# ============================================================
# Closed centerline tangents
# ============================================================

def closed_tangents(
    points,
):

    tangents = []

    n = len(
        points
    )


    for i in range(n):

        previous_point = (
            points[
                (i - 1) % n
            ]
        )


        next_point = (
            points[
                (i + 1) % n
            ]
        )


        tx = (
            next_point[0]
            - previous_point[0]
        )


        ty = (
            next_point[1]
            - previous_point[1]
        )


        tangents.append(
            normalize(
                tx,
                ty,
            )
        )


    return (
        tangents
    )


# ============================================================
# Ribbon geometry
# ============================================================

def ribbon_polygon(
    points,
    tangents,
    widths,
):

    left = []
    right = []


    for (
        (x, y),
        (tx, ty),
        width,
    ) in zip(
        points,
        tangents,
        widths,
    ):

        nx = -ty
        ny = tx

        half = (
            width / 2
        )


        left.append(
            (
                x + nx * half,
                y + ny * half,
            )
        )


        right.append(
            (
                x - nx * half,
                y - ny * half,
            )
        )


    return (
        left
        + right[::-1]
    )


def polygon_to_path(
    polygon,
):

    commands = [
        (
            f"M "
            f"{polygon[0][0]:.3f} "
            f"{polygon[0][1]:.3f}"
        )
    ]


    for x, y in polygon[1:]:

        commands.append(
            (
                f"L "
                f"{x:.3f} "
                f"{y:.3f}"
            )
        )


    commands.append(
        "Z"
    )


    return (
        " ".join(
            commands
        )
    )


# ============================================================
# Blue orbit + white underlay
# ============================================================

def build_orbit_paths():

    points, widths = (
        build_centerline()
    )


    tangents = (
        closed_tangents(
            points
        )
    )


    # Blue track
    blue_polygon = (
        ribbon_polygon(
            points,
            tangents,
            widths,
        )
    )


    # Same track, only wider
    white_widths = [
        (
            width
            + 2 * ORBIT_BORDER
        )
        for width in widths
    ]


    white_polygon = (
        ribbon_polygon(
            points,
            tangents,
            white_widths,
        )
    )


    return (
        polygon_to_path(
            white_polygon
        ),

        polygon_to_path(
            blue_polygon
        ),
    )


# ============================================================
# Serif I
# ============================================================

def create_i_path():

    stem_left = (
        CX
        - I_STEM / 2
    )

    stem_right = (
        CX
        + I_STEM / 2
    )


    top = 145
    bottom = 875


    serif_left = 378
    serif_right = 652


    shoulder = 60
    cap_h = 28


    return f"""
    M {serif_left} {top}

    L {serif_right} {top}

    L {serif_right} {top + cap_h}

    C
      {stem_right + shoulder}
      {top + cap_h},

      {stem_right + 22}
      {top + 54},

      {stem_right}
      {top + 96}

    L
      {stem_right}
      {bottom - 96}

    C
      {stem_right + 22}
      {bottom - 54},

      {stem_right + shoulder}
      {bottom - cap_h},

      {serif_right}
      {bottom - cap_h}

    L
      {serif_right}
      {bottom}

    L
      {serif_left}
      {bottom}

    L
      {serif_left}
      {bottom - cap_h}

    C
      {stem_left - shoulder}
      {bottom - cap_h},

      {stem_left - 22}
      {bottom - 54},

      {stem_left}
      {bottom - 96}

    L
      {stem_left}
      {top + 96}

    C
      {stem_left - 22}
      {top + 54},

      {stem_left - shoulder}
      {top + cap_h},

      {serif_left}
      {top + cap_h}

    Z
    """


# ============================================================
# SVG construction
# ============================================================

def build_svg():

    (
        orbit_white,
        orbit_blue,
    ) = build_orbit_paths()


    dot_x, dot_y = (
        ellipse_point(
            DOT_ANGLE
        )
    )


    i_path = (
        create_i_path()
    )


    return f"""<?xml version="1.0" encoding="UTF-8"?>

<svg
    xmlns="http://www.w3.org/2000/svg"
    width="{SIZE}"
    height="{SIZE}"
    viewBox="0 0 {SIZE} {SIZE}"
>

    <defs>

        <clipPath id="frontClip">

            <rect
                x="0"
                y="{FRONT_CLIP_Y}"
                width="{SIZE}"
                height="{SIZE - FRONT_CLIP_Y}"
            />

        </clipPath>

    </defs>


    <!-- ==================================================
         Z0
         WHITE ORBIT UNDERLAY
         ================================================== -->

    <path
        d="{orbit_white}"
        fill="{WHITE}"
    />


    <!-- ==================================================
         Z1
         BLUE ORBIT
         ================================================== -->

    <path
        d="{orbit_blue}"
        fill="{BLUE}"
    />


    <!-- ==================================================
         Z2
         WHITE I UNDERLAY
         ================================================== -->

    <path
        d="{i_path}"
        fill="{WHITE}"
        stroke="{WHITE}"
        stroke-width="{2 * I_BORDER}"
        stroke-linejoin="round"
    />


    <!-- ==================================================
         Z3
         GOLD I
         ================================================== -->

    <path
        d="{i_path}"
        fill="{GOLD}"
    />


    <!-- ==================================================
         Z4
         LOWER WHITE ORBIT
         ================================================== -->

    <g
        clip-path="url(#frontClip)"
    >

        <path
            d="{orbit_white}"
            fill="{WHITE}"
        />


        <!-- ==============================================
             Z5
             LOWER BLUE ORBIT
             ============================================== -->

        <path
            d="{orbit_blue}"
            fill="{BLUE}"
        />

    </g>


    <!-- ==================================================
         Z6
         WHITE NODE UNDERLAY
         ================================================== -->

    <circle
        cx="{dot_x:.3f}"
        cy="{dot_y:.3f}"
        r="{DOT_DIAMETER / 2 + DOT_BORDER}"
        fill="{WHITE}"
    />


    <!-- ==================================================
         Z7
         GOLD NODE
         ================================================== -->

    <circle
        cx="{dot_x:.3f}"
        cy="{dot_y:.3f}"
        r="{DOT_DIAMETER / 2}"
        fill="{GOLD}"
    />

</svg>
"""


# ============================================================
# Export
# ============================================================

def export_svg(
    svg_text,
):

    path = (
        OUTDIR
        / "imasen-phi.svg"
    )


    path.write_text(
        svg_text,
        encoding="utf-8",
    )


    return path


def export_master_png(
    svg_text,
):

    path = (
        OUTDIR
        / "imasen-phi-1024.png"
    )


    cairosvg.svg2png(
        bytestring=(
            svg_text.encode(
                "utf-8"
            )
        ),
        write_to=str(
            path
        ),
        output_width=1024,
        output_height=1024,
    )


    return (
        path
    )


def export_png_sizes(
    master_path,
):

    master = (
        Image
        .open(
            master_path
        )
        .convert(
            "RGBA"
        )
    )


    for size in [
        512,
        256,
        128,
        64,
        48,
        32,
        16,
    ]:

        image = (
            master.resize(
                (
                    size,
                    size,
                ),
                Image.Resampling.LANCZOS,
            )
        )


        image.save(
            OUTDIR
            / f"imasen-phi-{size}.png"
        )


def export_ico(
    master_path,
):

    master = (
        Image
        .open(
            master_path
        )
        .convert(
            "RGBA"
        )
    )


    master.save(
        OUTDIR
        / "favicon.ico",
        format="ICO",
        sizes=[
            (16, 16),
            (32, 32),
            (48, 48),
            (64, 64),
            (128, 128),
            (256, 256),
        ],
    )


# ============================================================
# Main
# ============================================================

def main():

    svg_text = (
        build_svg()
    )


    svg_path = (
        export_svg(
            svg_text
        )
    )


    png_path = (
        export_master_png(
            svg_text
        )
    )


    export_png_sizes(
        png_path
    )


    export_ico(
        png_path
    )


    print(
        "Generated files:"
    )


    print(
        "SVG:",
        svg_path.resolve(),
    )


    print(
        "PNG:",
        png_path.resolve(),
    )


    print(
        "ICO:",
        (
            OUTDIR
            / "favicon.ico"
        ).resolve(),
    )


if __name__ == "__main__":
    main()

Generated files:
SVG: E:\Documents\Desktop\imasen_phi_logo_v6\imasen_phi_logo_v6\imasen_phi_logo_v7\imasen-phi.svg
PNG: E:\Documents\Desktop\imasen_phi_logo_v6\imasen_phi_logo_v6\imasen_phi_logo_v7\imasen-phi-1024.png
ICO: E:\Documents\Desktop\imasen_phi_logo_v6\imasen_phi_logo_v6\imasen_phi_logo_v7\favicon.ico
